In [ ]:
# FP-Growth Algorithm

"""FP-Growth (Frequent Pattern Growth) is an association rule mining algorithm used to find frequent itemsets in a transactional dataset.
It is an alternative to the Apriori algorithm. The major difference is that Apriori generates candidate itemsets and repeatedly scans the database, whereas FP-Growth compresses the transaction database into an FP-Tree (Frequent Pattern Tree) and mines frequent patterns from that tree.

Why do we use FP-Growth?
Apriori can become expensive when the dataset is large because it generates many candidate itemsets.

FP-Growth improves this by:
1. Scanning the database to calculate item frequencies.
2. Removing items that do not satisfy minimum support.
3. Ordering the remaining items according to their frequency.
4. Building an FP-Tree.
5. Mining the FP-Tree to find frequent itemsets.
6. Generating association rules using minimum confidence.

Important Parameters
For our dataset:
Minimum Support = 50%
Minimum Confidence = 50%
Number of transactions = 5
Therefore, the minimum support count is: 5×50%=2.5
Since an item cannot occur in half a transaction, an itemset must occur at least 3 times."""

In [ ]:
"""
Given Dataset
| Transaction | Items            |
| ----------- | ---------------- |
| T1          | E, K, M, N, O, Y |
| T2          | D, E, K, N, O, Y |
| T3          | C, K, M, U, Y    |
| T4          | C, E, I, K, Z, O |
| T5          | A, E, K, M, N    |

Step 1: Count Frequency of Every Item
We first scan the database and count every item.
| Item | Count | Support |
| ---- | ----: | ------: |
| A    |     1 |     20% |
| C    |     2 |     40% |
| D    |     1 |     20% |
| E    |     4 |     80% |
| I    |     1 |     20% |
| K    |     5 |    100% |
| M    |     3 |     60% |
| N    |     3 |     60% |
| O    |     3 |     60% |
| U    |     1 |     20% |
| Y    |     3 |     60% |
| Z    |     1 |     20% |
Minimum support count = 3.
Therefore, remove:A, C, D, I, U, Z
because their frequency is less than 3.
The frequent items are: K,E,M,N,O,Y

Step 2: Arrange Frequent Items
FP-Growth normally arranges items in descending order of frequency.
Here:
K = 5
E = 4
M = 3
N = 3
O = 3
Y = 3

For items having the same frequency, we need a consistent tie-breaking order. Let's use:
K>E>M>N>O>Y
Therefore, every transaction will be reordered according to this order.

Step 3: Remove Infrequent Items and Reorder Transactions
T1
Original:
E, K, M, N, O, Y

After removing infrequent items and sorting:
K,E,M,N,O,Y

| Transaction | Ordered Frequent Items |
| ----------- | ---------------------- |
| T1          | K, E, M, N, O, Y       |
| T2          | K, E, N, O, Y          |
| T3          | K, M, Y                |
| T4          | K, E, O                |
| T5          | K, E, M, N             |

Step 4: Build the FP-Tree
We now insert each transaction into the FP-Tree.
The root is represented by: NULL

Insert T1
T1:
K → E → M → N → O → Y
The tree initially becomes:
Root
 |
 K:1
 |
 E:1
 |
 M:1
 |
 N:1
 |
 O:1
 |
 Y:1

Insert T2
T2:K → E → N → O → Y
K and E already exist, so their counts increase.

Root
 |
 K:2
 |
 E:2
 / \
M:1 N:1
 |   |
N:1 O:1
 |   |
O:1 Y:1
 |
Y:1

Insert T3 and so on 
The final FP-Tree can be represented as:
                         Root
                           |
                          K:5
                         /   \
                      E:4    M:1
                     /   \      \
                  M:2    N:1    Y:1
                  |       |
                 N:2     O:1
                /  \       |
             O:1   Y:1    Y:1

                     E branch
                  also contains
                     O:1

The exact tree visualization can be drawn more clearly by following the branches:
Root
 |
 K:5
 ├── E:4
 │   ├── M:2
 │   │   └── N:2
 │   │       └── O:1
 │   │           └── Y:1
 │   │
 │   ├── N:1
 │   │   └── O:1
 │   │       └── Y:1
 │   │
 │   └── O:1
 │
 └── M:1
     └── Y:1

Step 5: Mine the FP-Tree

Now FP-Growth starts mining the tree.

We generally start with the least frequent items and work upward.

Our items have frequencies:
| Item | Frequency |
| ---- | --------: |
| Y    |         3 |
| O    |         3 |
| N    |         3 |
| M    |         3 |
| E    |         4 |
| K    |         5 |
Mining Y : Y occurs in:

T1
T2
T3

Therefore: Support(Y)=3/5=60%

So: Y is frequent.
Now we look at the paths leading to Y.
These paths allow us to find combinations involving Y.
From the transactions:
T1 → K,E,M,N,O,Y
T2 → K,E,N,O,Y
T3 → K,M,Y
We can check possible combinations.

KY
K and Y occur together in T1, T2, T3.
Count = 3
Support(KY)=60%
Therefore: KY is frequent.

Other combinations such as EY, MY, NY, OY occur fewer than 3 times, so they are not frequent.

Mining OO occurs in:
T1
T2
T4

Therefore: Support(O)=60%
Now check combinations. KO
K and O occur together in T1, T2 and T4.
Support(KO)=3/5=60%
Therefore:KO is frequent.

EO
E and O occur together in T1, T2 and T4.
Support(EO)=60%
Therefore: EO is frequent.
Other combinations such as MO, NO, YO occur fewer than 3 times.

Mining N: N occurs in:
T1
T2
T5
Therefore: Support(N)=60%

KN
K and N occur in T1, T2 and T5.
Support(KN)=60%
Therefore:KN is frequent.

EN
E and N occur in T1, T2 and T5.
Support(EN)=60%
Therefore: EN is frequent.
Other combinations involving N do not reach the minimum support.

Mining M: M occurs in:
T1
T3
T5
Therefore:Support(M)=60%

KM
K and M occur in:
T1
T3
T5

Therefore:Support(KM)=60%
So: KM is frequent.

But:
EM occurs only in T1 and T5 → 40%
MN occurs only in T1 and T5 → 40%
MY occurs only in T1 and T3 → 40%
Therefore, they are not frequent.

Mining E
E occurs in:
T1
T2
T4
T5
Therefore:
Support(E)=4/5=80%

EK
E and K occur in all four E transactions:
Support(EK)=80%
Therefore: EK is frequent.

Step 6: Find Frequent 3-Itemsets
Now we check combinations of three frequent items.

EKN
E, K and N occur together in:
T1
T2
T5
Count = 3
# Support(EKN)=3/5=60%
Therefore: EKN  is frequent.

EKO
E, K and O occur together in:
T1
T2
T4

Count = 3

Support(EKO)=60%
Therefore: EKO

is frequent.
Other 3-item combinations do not reach the minimum support.

Final Frequent Itemsets from FP-Growth
The result is:
1-Itemsets
E,K,M,N,O,Y
2-Itemsets
EK,EN,EO,KM,KN,KO,KY
3-Itemsets
EKN,EKO
4-Itemsets
None
Notice that the frequent itemsets obtained are the same as Apriori. This is expected because both algorithms are solving the same frequent-itemset mining problem. The difference is mainly in how they search for those itemsets.

Step 7: Generate Association Rules
Now we apply:
Minimum Confidence=50%
For example, from:
EKN
we can generate:
E → K,N
Confidence(E→KN)= Support(EKN)/ Support(E) = 60/80 = 75%
✅ Strong rule.

K → E,N
Confidence(K→EN)= 60/60=100%
✅ Strong rule.

N → E,K
Confidence(N→EK)= 60/60=100%
✅ Strong rule.

Similarly, from EKO:
E → K,O     75/100=75%
K → E,O    60/100=60%
O → E,K   60/60=100%
"""

In [ ]:
"""| Feature              | Apriori                     | FP-Growth                            |
| -------------------- | --------------------------- | ------------------------------------ |
| Main idea            | Generate candidate itemsets | Build FP-Tree                        |
| Candidate generation | Yes                         | No explicit candidate generation     |
| Database scans       | Multiple scans              | Usually two major scans              |
| Data structure       | Itemsets                    | FP-Tree                              |
| Large datasets       | Can be expensive            | Generally more efficient             |
| Pruning              | Uses Apriori property       | Uses conditional pattern bases/trees |
| Result               | Frequent itemsets           | Same frequent itemsets               |
"""

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 

In [2]:
pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 500)

In [3]:
dataset = pd.read_csv("groceries - groceries.csv", sep=";")
dataset.head(3)

,Item 1,Item 2,Item 3,Item 4,Item 5,Item 6,Item 7,Item 8,Item 9,Item 10,Item 11,Item 12,Item 13,Item 14,Item 15,Item 16,Item 17,Item 18,Item 19,Item 20,Item 21,Item 22,Item 23,Item 24,Item 25,Item 26,Item 27,Item 28,Item 29,Item 30,Item 31,Item 32
0,citrus fruit,semi-finished bread,margarine,ready soups,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,tropical fruit,yogurt,coffee,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,whole milk,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
dataset.shape

(9835, 32)

In [5]:
type(dataset.iloc[0, 6]) , type(dataset.iloc[0, 1])

(float, str)

In [6]:
market = []
for i in range(0,dataset.shape[0]):
    cus=[]
    for j in dataset.columns:
        # if type(dataset[j][i]==str):  # nan type -- float an other data --str to delete nan we delete the float type data
        if isinstance(dataset.loc[i, j], str):
            cus.append(dataset[j][i])
    market.append(cus)

In [7]:
l = []
for i in market:
    for j in i :
        l.append(j)

In [8]:
import collections
p = collections.Counter(l)
d = {"Item Name":p.keys(),"values":p.values()}
pd.DataFrame(d).sort_values(by=["values"],ascending = False)

,Item Name,values
7,whole milk,2513
11,other vegetables,1903
17,rolls/buns,1809
31,soda,1715
5,yogurt,1372
24,bottled water,1087
42,root vegetables,1072
4,tropical fruit,1032
52,shopping bags,969
50,sausage,924


In [9]:
from mlxtend.preprocessing.transactionencoder import TransactionEncoder
tr = TransactionEncoder()
tr.fit(market)
df = pd.DataFrame(tr.transform(market),columns=tr.columns_)

In [11]:
df

,Instant food products,UHT-milk,abrasive cleaner,artif. sweetener,baby cosmetics,baby food,bags,baking powder,bathroom cleaner,beef,berries,beverages,bottled beer,bottled water,brandy,brown bread,butter,butter milk,cake bar,candles,candy,canned beer,canned fish,canned fruit,canned vegetables,cat food,cereals,chewing gum,chicken,chocolate,chocolate marshmallow,citrus fruit,cleaner,cling film/bags,cocoa drinks,coffee,condensed milk,cooking chocolate,cookware,cream,cream cheese,curd,curd cheese,decalcifier,dental care,dessert,detergent,dish cleaner,dishes,dog food,domestic eggs,female sanitary products,finished products,fish,flour,flower (seeds),flower soil/fertilizer,frankfurter,frozen chicken,frozen dessert,frozen fish,frozen fruits,frozen meals,frozen potato products,frozen vegetables,fruit/vegetable juice,grapes,hair spray,ham,hamburger meat,hard cheese,herbs,honey,house keeping products,hygiene articles,ice cream,instant coffee,jam,ketchup,kitchen towels,kitchen utensil,light bulbs,liqueur,liquor,liquor (appetizer),liver loaf,long life bakery product,make up remover,male cosmetics,margarine,mayonnaise,meat,meat spreads,misc. beverages,mustard,napkins,newspapers,nut snack,nuts/prunes,oil,onions,organic products,organic sausage,other vegetables,packaged fruit/vegetables,pasta,pastry,pet care,photo/film,pickled vegetables,pip fruit,popcorn,pork,potato products,potted plants,preservation products,processed cheese,prosecco,pudding powder,ready soups,red/blush wine,rice,roll products,rolls/buns,root vegetables,rubbing alcohol,rum,salad dressing,salt,salty snack,sauces,sausage,seasonal products,semi-finished bread,shopping bags,skin care,sliced cheese,snack products,soap,soda,soft cheese,softener,sound storage medium,soups,sparkling wine,specialty bar,specialty cheese,specialty chocolate,specialty fat,specialty vegetables,spices,spread cheese,sugar,sweet spreads,syrup,tea,tidbits,toilet cleaner,tropical fruit,turkey,vinegar,waffles,whipped/sour cream,whisky,white bread,white wine,whole milk,yogurt,zwieback
0,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,Tr

In [10]:
from mlxtend.frequent_patterns import fpgrowth
fpgrowth(df,min_support = 0.07,use_colnames=True,max_len=3).sort_values(by=["support"])

,support,itemsets
17,0.071683,(whipped/sour cream)
11,0.072293,(fruit/vegetable juice)
18,0.074835,"(other vegetables, whole milk)"
4,0.075648,(pip fruit)
14,0.077682,(canned beer)
10,0.079817,(newspapers)
7,0.080529,(bottled beer)
0,0.082766,(citrus fruit)
12,0.088968,(pastry)
15,0.093950,(sausage)
